In [ ]:
import pandas as pd
import numpy as np

# Библиотеки для визуализации
import matplotlib.pyplot as plt
import seaborn as sns

# Библиотеки для машинного обучения
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

In [ ]:
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
try:
    df = pd.read_csv('./insurance.csv')
except:
    sample_data = {
        'age': [19, 18, 28, 33, 32, 31, 46, 37, 37, 60],
        'sex': ['female', 'male', 'male', 'male', 'male', 'female', 'female', 'female', 'male', 'female'],
        'bmi': [27.9, 33.77, 33.0, 22.705, 28.88, 25.74, 33.44, 27.74, 29.83, 25.84],
        'children': [0, 1, 3, 0, 0, 0, 1, 3, 2, 0],
        'smoker': ['yes', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no', 'no'],
        'region': ['southwest', 'southeast', 'southeast', 'northwest', 'northwest', 'southeast', 'southeast', 'northwest', 'northeast', 'northwest'],
        'charges': [16884.924, 1725.5523, 4449.462, 21984.47061, 3866.8552, 756.6216, 8240.5896, 7281.5056, 6406.4107, 28923.13692]
    }

    df = pd.DataFrame(sample_data)

In [ ]:
print(f"Размер датасета: {df.shape[0]} строк, {df.shape[1]} столбцов")

missing_values = df.isnull().sum()
print("Пропущенные значения в каждом столбце:")
print(missing_values)

In [ ]:
df_processed = df.copy()
le = LabelEncoder()
df_processed['sex_encoded'] = le.fit_transform(df_processed['sex'])
print("Кодирование пола:", dict(zip(le.classes_, le.transform(le.classes_))))
df_processed['smoker_encoded'] = le.fit_transform(df_processed['smoker'])
print("Кодирование курения:", dict(zip(le.classes_, le.transform(le.classes_))))
region_dummies = pd.get_dummies(df_processed['region'], prefix='region').astype(int)
df_processed = pd.concat([df_processed, region_dummies], axis=1).drop(['sex','smoker','region'], axis=1)

In [ ]:
# 1.1. Постройте график зависимости стоимости страховки от возраста отдельно для курящих и некурящих
plt.figure(figsize=(10,6))
for s, col in [('no','tab:blue'),('yes','tab:red')]:
    sub = df[df.smoker==s]
    plt.scatter(sub.age, sub.charges, label=f'Курение: {s}', alpha=0.5, color=col)
    p = np.poly1d(np.polyfit(sub.age, sub.charges, 1))
    xs = np.sort(sub.age.unique())
    plt.plot(xs, p(xs), color=col, lw=2)
plt.xlabel('Возраст')
plt.ylabel('Стоимость страховки')
plt.title('Зависимость стоимости от возраста (курящие vs некурящие)')
plt.legend()
plt.show()

In [ ]:
# 1.2. Средний ИМТ по регионам
print(df.groupby('region')['bmi'].mean().sort_values(ascending=False).round(3))

In [ ]:
# 1.3. Возраст с максимальной стоимостью
max_charge_age = df.loc[df['charges'].idxmax(), 'age']
max_charge_value = df['charges'].max()
print(f"Максимальная стоимость ${max_charge_value:.2f} в возрасте {max_charge_age} лет")

In [ ]:
# 2.1. Попробуйте создать новый признак: age_group (молодые: 18-30, средние: 31-50, старшие: 51+)
df2 = df_processed.copy()
df2['age_group'] = pd.cut(df2['age'], bins=[17,30,50,100], labels=['young','middle','senior'])
print(df2['age_group'].value_counts())
age_dummies = pd.get_dummies(df2['age_group'], prefix='age_group').astype(int)
df2 = pd.concat([df2, age_dummies], axis=1).drop('age_group', axis=1)

In [ ]:
# 2.2. Создайте признак high_bmi (1 если ИМТ > 30, иначе 0)
df2['high_bmi'] = (df2['bmi'] > 30).astype(int)
print(df2.columns.tolist())

In [ ]:
# 2.3. Обучите модель с новыми признаками и сравните R²
def evaluate(data, name):
    X_ = data.drop('charges', axis=1)
    y_ = data['charges']
    Xtr, Xte, ytr, yte = train_test_split(X_, y_, test_size=0.2, random_state=42)
    m = LinearRegression().fit(Xtr, ytr)
    p = m.predict(Xte)
    return {'Модель': name, 'Признаков': X_.shape[1], 'R²': r2_score(yte,p), 'MAE': mean_absolute_error(yte,p), 'RMSE': np.sqrt(mean_squared_error(yte,p))}, m, X_

res = []
r, m_base, _ = evaluate(df_processed, 'Базовая'); res.append(r)
r, m_new, X_new = evaluate(df2, '+ age_group + high_bmi'); res.append(r)
df3 = df2.copy()
df3['smoker_high_bmi'] = df3['smoker_encoded']*df3['high_bmi']
r, m_int, _ = evaluate(df3, '+ smoker×high_bmi'); res.append(r)
results = pd.DataFrame(res)
print(results.round(4).to_string(index=False))
print(f"\nИзменение R² (новые признаки vs базовая): {res[1]['R²']-res[0]['R²']:+.4f}")
print(f"Изменение R² (с взаимодействием vs базовая): {res[2]['R²']-res[0]['R²']:+.4f}")
print("\nКоэффициенты модели с новыми признаками:")
print(pd.Series(m_new.coef_, index=X_new.columns).round(2).sort_values(key=abs, ascending=False))

### 3.1. Почему курение так сильно влияет на стоимость страховки
* **Риск.** Курение повышает вероятность дорогих в лечении заболеваний (сердечно-сосудистые, онкологические, лёгочные), поэтому ожидаемые выплаты компании по таким клиентам выше, а цена полиса отражает ожидаемые выплаты.
* **Данные.** Средняя стоимость у курящих — около `32 050`, у некурящих — около `8 434` (в 3,8 раза выше); корреляция `smoker` и `charges` = 0,787, что намного больше, чем у любого другого признака. В модели курение добавляет ≈ $23 650 при прочих равных.
* **Распределения почти не пересекаются:** минимум у курящих ($12 829) выше средней стоимости некурящих.
* **Курение усиливает другие факторы.** У некурящих высокий ИМТ почти не меняет цену (≈  `7 967` → `8 853`), у курящих — удваивает (≈  `21 369` → `41 693`). Линейная модель такой эффект «не видит», пока не добавлено взаимодействие.
* _Оговорка_: это ассоциация в данных, а не строгое доказательство причинности.

### 3.2. Три способа улучшить модель
1. **Признак-взаимодействие smoker × high_bmi** (или smoker × bmi). Проверено в задании 2.3: R² вырос с 0,784 до 0,885, MAE упал с `4181` до `2 373` — самое большое улучшение.
1. **Преобразование целевой переменной** (log(charges)) или нелинейные члены (age²). Распределение `charges` скошено вправо (среднее  `13 270` > медиана `9 382`, 10% выбросов по IQR), остатки неоднородны — линейная модель на сырых значениях этим нарушает допущения. Альтернатива — отдельные модели для курящих и некурящих.
2. **Нелинейные модели и честная валидация:** Random Forest / Gradient Boosting, кросс-валидация вместо одного разбиения 80/20, подбор гиперпараметров; также стандартизация признаков, чтобы коэффициенты были сопоставимы.

### 3.3. Какие дополнительные данные собрать
* **Медицинские:** хронические заболевания, история обращений и госпитализаций, семейный анамнез, давление и холестерин.
* **Образ жизни:** стаж и интенсивность курения (пачко-лет), алкоголь, физическая активность, питание.
* **Социально-экономические:** профессия (риск травм), доход, уровень образования.
* **Страховые:** история страховых случаев, тип и покрытие полиса, франшиза.
* Точный адрес/город вместо крупного региона и доступность медицинской помощи.